# Madden Franchise Editor

Cuts a long Madden franchise recording into an edited episode.

**Run the cells in order, top to bottom.** Click the ▶ button on the left of each
cell, wait for the green checkmark, then move to the next one.

---

### Before you start: turn on the free GPU

This makes the slowest step about 20x faster. Do it once, now:

1. Menu bar → **Runtime** → **Change runtime type**
2. Under *Hardware accelerator* pick **T4 GPU**
3. Click **Save**

If no GPU is available (free tier runs out sometimes) everything still works,
just slower. Cell 1 will tell you which you got.

## Cell 1 — Setup

Installs everything. Takes 2–3 minutes. Run once per session.

In [ ]:
#@title Run me first { display-mode: "form" }
import subprocess, sys, os, textwrap

REPO = "https://github.com/t06726495-afk/CluadeAnimator.git"
BRANCH = "claude/video-review-119k2h"

print("Checking hardware...")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    print(f"  GPU: {gpu.stdout.strip()}  (transcription will be fast)")
else:
    print("  No GPU. Everything still works, transcription is just slower.")
    print("  To fix: Runtime > Change runtime type > T4 GPU, then re-run this cell.")

print("\nInstalling ffmpeg...")
subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg",
               shell=True, check=True)

print("Installing whisper (this is the slow part of setup)...")
subprocess.run(f"{sys.executable} -m pip install -q openai-whisper",
               shell=True, check=True)

print("\nGetting the editing skill...")
if not os.path.isdir("/content/CluadeAnimator"):
    clone = subprocess.run(
        f"git clone -q --branch {BRANCH} {REPO} /content/CluadeAnimator",
        shell=True, capture_output=True, text=True)
    if clone.returncode != 0:
        print(textwrap.dedent(f"""
            Could not download the skill automatically.
            {clone.stderr.strip()}

            The repository is probably private. Two ways forward:
              1. Make the repo public on github.com (Settings > General >
                 Danger Zone > Change visibility), then re-run this cell.
              2. Download the repo as a ZIP from github.com, upload it to your
                 Google Drive, and tell Claude — there's a Drive-based version
                 of this cell.
        """))
    else:
        print("  done")
else:
    print("  already downloaded")

os.makedirs("/content/work", exist_ok=True)
SKILL = "/content/CluadeAnimator/.claude/skills/madden-franchise-editor"
print(f"\nReady." if os.path.isdir(SKILL) else "\nSkill folder missing — see above.")

## Cell 2 — Connect your Google Drive

A popup will ask permission. Click through it and pick your Google account.

**Put your recording in Google Drive first.** On your Chromebook: open the Files
app, drag the video into Google Drive. Note the filename — you need it next.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob
print("\nVideos found in your Drive:\n")
vids = []
for ext in ("mp4", "mkv", "mov", "MP4", "MKV", "MOV"):
    vids += glob.glob(f"/content/drive/MyDrive/**/*.{ext}", recursive=True)
if vids:
    for v in sorted(vids)[:40]:
        size = os.path.getsize(v) / 1e9
        print(f"  {size:5.1f} GB   {v}")
else:
    print("  None found. Put your recording in Google Drive, wait for it to")
    print("  finish syncing, then run this cell again.")

## Cell 3 — Pick your video

Copy one of the paths printed above into the box, then run.

**Leave `test_mode` ticked the first time.** It runs on just the first 5 minutes
so you can check everything works before committing to a full episode.

In [ ]:
video_path = "/content/drive/MyDrive/myvideo.mp4"  #@param {type:"string"}
test_mode = True  #@param {type:"boolean"}
target_minutes = 22  #@param {type:"integer"}

import os, subprocess

if not os.path.exists(video_path):
    raise SystemExit(f"Not found: {video_path}\nCopy a path from Cell 2 exactly.")

dur = float(subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1:nokey=1", video_path],
    capture_output=True, text=True).stdout.strip())
print(f"Source: {dur/60:.1f} minutes, {os.path.getsize(video_path)/1e9:.1f} GB")

if test_mode:
    RAW = "/content/work/test.mp4"
    print("\nTest mode — making a 5 minute sample...")
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", video_path,
                    "-t", "300", "-c", "copy", RAW], check=True)
    print("Working on the 5 minute sample. Untick test_mode for the real run.")
else:
    RAW = video_path
    print(f"\nFull episode. Target length: {target_minutes} minutes.")

## Cell 4 — Speech to text

The slowest step. With a GPU: roughly 3–6 minutes for a full episode. Without
one: closer to an hour.

It prints nothing for a while — that's normal, it's working.

In [ ]:
!cd /content && python3 {SKILL}/scripts/transcribe.py "{RAW}" \
    -o /content/work/transcript.json --backend local --model base.en

## Cell 5 — Work out the structure

**This is the one to actually read.** It prints every section it found in your
recording — cold open, roster moves, gameplay, postgame and so on.

You want that list to roughly match your real episode. If it's one giant
segment, or the labels look nonsense, stop and send me the output — that's a
fixable pattern-matching problem, and nothing after this will be right until
it's fixed.

In [ ]:
!cd /content && python3 {SKILL}/scripts/analyze.py "{RAW}" \
    /content/work/transcript.json -o /content/work/analysis.json

## Cell 6 — Decide the cuts, then check them

Builds the cut list and then runs it past the style rules — it will complain if
it cut a catchphrase, made the pacing robotic, or left a sponsor read in.

In [ ]:
target = [] if test_mode else ["--target", str(target_minutes)]

!cd /content && python3 {SKILL}/scripts/build_edl.py \
    /content/work/analysis.json -o /content/work/edl.json {" ".join(target)}

print("\n" + "=" * 60 + "\nQUALITY CHECK\n" + "=" * 60)
!cd /content && python3 {SKILL}/scripts/qa.py \
    /content/work/edl.json /content/work/analysis.json || true

## Cell 7 — Render

Cuts the actual video files. The second slowest step — roughly 10–20 minutes for
a full episode.

Produces the rough cut, numbered clips for CapCut, captions, and edit notes.

In [ ]:
!cd /content && python3 {SKILL}/scripts/render.py \
    /content/work/edl.json /content/work/analysis.json -o /content/out

## Cell 8 — Read the edit notes

What got cut and why, your highlight moments ranked by how excited you sounded,
comedy candidates, and the list of things still to do by hand in CapCut.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/content/out/edit_notes.md").read()))

## Cell 9 — Send it back to Google Drive

**Do not skip this.** Colab wipes everything when the session ends. This copies
your finished pieces to Drive, where your Chromebook can see them.

Afterwards: Files app → Google Drive → `MaddenEdits` → open in CapCut.

In [ ]:
import shutil, datetime, os

stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
tag = "TEST" if test_mode else "episode"
dest = f"/content/drive/MyDrive/MaddenEdits/{tag}_{stamp}"

print(f"Copying to {dest} ...")
shutil.copytree("/content/out", dest)

total = sum(os.path.getsize(os.path.join(r, f))
            for r, _, fs in os.walk(dest) for f in fs)
print(f"\nDone — {total/1e9:.2f} GB copied.")
print("\nOn your Chromebook: Files app > Google Drive > MaddenEdits")
print("  rough_cut.mp4  — the assembled edit, watch this first")
print("  clips/         — numbered in order, drag into CapCut")
print("  captions.srt   — import into CapCut")
print("  edit_notes.md  — what to finish by hand")

---

## Doing a real episode

Once the test looks right:

1. Go back to **Cell 3**, untick **test_mode**, set **target_minutes**
2. Re-run Cells 3 through 9

No need to redo Cells 1 and 2 unless the session disconnected.

## If something goes wrong

| Problem | Fix |
|---|---|
| "Session disconnected" | Colab times out when idle. Re-run from Cell 1. |
| Cell 1 can't download the skill | The repo is private — see the message it printed. |
| Cell 2 lists no videos | Video hasn't finished syncing to Drive yet. |
| Cell 5's sections look wrong | Stop and send me the output. |
| Edit is too choppy | Cell 6: add `--play-floor 0.3` |
| Edit is too long | Cell 3: lower `target_minutes` |

Colab's free tier has daily limits. If you hit one, it comes back the next day.